# v4 Notebook B - experts (Llama-Guard-3-8B, Granite-Guardian-4.1-8B)
Self-provisions data (no wiring to A). **Main run = L0 view only for both experts** -> full headline (H1/H3/H4/H6). Per-shard progress in the Logs tab; **shard-checkpointed and resumable** - if a session times out, just re-run (commit) and it continues. Output: `v4_B_experts.npz`. Cell 3 (raw view) is OPTIONAL, run it in a later session only after the L0 run finishes.

In [ ]:
# Cell 1 - setup (Kaggle T4; Internet ON; Secrets: HF_TOKEN, AGENT_API_KEY)
import os, sys, subprocess, glob
KAGGLE = os.path.isdir('/kaggle/working')
DEST = ('/kaggle/working' if KAGGLE else '/content') + '/vyuha_src'
if not os.path.isdir(f'{DEST}/.git'):
    subprocess.run(['git','clone','--depth','1','https://github.com/g25ait2149/vyuha.git',DEST], check=False)
else:
    subprocess.run(['git','-C',DEST,'pull','--ff-only'], check=False)
hits = glob.glob(DEST+'/**/vyuha/__init__.py', recursive=True)
assert hits, 'clone failed - turn Internet ON'
ROOT = os.path.dirname(os.path.dirname(hits[0])); sys.path.insert(0, ROOT); os.chdir(ROOT)
subprocess.run('pip -q install -U "bitsandbytes>=0.46.1" accelerate "transformers>=4.44" datasets scikit-learn sentencepiece "protobuf<5" huggingface_hub', shell=True)
tok = ''
try:
    if KAGGLE:
        from kaggle_secrets import UserSecretsClient; tok = UserSecretsClient().get_secret('HF_TOKEN')
    else:
        from google.colab import userdata; tok = userdata.get('HF_TOKEN')
except Exception as e:
    print('HF_TOKEN missing:', repr(e)[:100])
os.environ['HF_TOKEN'] = os.environ['HUGGINGFACE_HUB_TOKEN'] = tok or ''
from huggingface_hub import login
if tok: login(tok)
import torch; print('GPUs:', torch.cuda.device_count(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
OUT = ('/kaggle/working' if KAGGLE else '/content') + '/v4'; os.makedirs(OUT, exist_ok=True)
print('OUT =', OUT)

In [ ]:
# Cell 2 - two experts x two views, one expert per GPU (T4 x2). Reads the pool texts saved by notebook A.
import json, os, numpy as np, torch
from vyuha.normalize.normalize import normalize
from vyuha.cascade.guards import load_model, forced_prefix, bucketed_scores
from vyuha.cascade.data import ensure_files
ensure_files(OUT)                                   # self-provision A's data/pool if not already present
EVAL = json.load(open(f'{OUT}/v4_data.json'))['eval_seeds']
WC_CALIB = json.load(open(f'{OUT}/v4_data.json'))['wc_calib']
WC_TEST  = json.load(open(f'{OUT}/v4_data.json'))['wc_test']
HARD     = json.load(open(f'{OUT}/v4_data.json'))['hard']
POOL = json.load(open(f'{OUT}/v4_pool.json'))['texts']; NS, NC = len(POOL), 60
flat = [t for row in POOL for t in row]
V = lambda xs: [normalize(t, full=True, version=2) for t in xs]
probes = [t for t in EVAL+WC_CALIB if len(t)<300][:5]
OUTF = f'{OUT}/v4_B_experts.npz'; store = dict(np.load(OUTF)) if os.path.exists(OUTF) else {}
import time
def save(): np.savez(OUTF, **store)

def score_key(tok, m, kind, pre, key, texts, shard=2000):
    """Resumable + checkpointed: score `texts` in shards, saving each shard to the npz so a timeout/restart
    continues from the last finished shard instead of redoing the whole key. Prints per-shard timing."""
    if key in store: print(f'  [resume] {key} ({len(store[key])})', flush=True); return store[key]
    parts = []
    for i in range(0, len(texts), shard):
        sk = f'{key}__s{i}'
        if sk in store: parts.append(store[sk]); continue
        t0 = time.time()
        arr = bucketed_scores(tok, m, texts[i:i+shard], kind, pre)
        store[sk] = arr; save(); parts.append(arr)
        print(f'  {key} {min(i+shard,len(texts))}/{len(texts)}  ({time.time()-t0:.0f}s/shard)', flush=True)
    full = np.concatenate(parts) if parts else np.zeros(0)
    store[key] = full
    for i in range(0, len(texts), shard): store.pop(f'{key}__s{i}', None)
    save(); return full

def score_expert(mid, kind, tag, device, views=('l0',)):
    final = f'{tag}/{views[-1]}/pool'
    if final in store: print('[resume] whole', tag, views, 'already done', flush=True); return
    tok, m = load_model(mid, True, device=device)
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()): pre = forced_prefix(tok, m, kind, probes)
    print(f'{tag}: model loaded, scoring views {views}...', flush=True)
    for view in views:
        ev = (lambda xs: V(xs)) if view == 'l0' else (lambda xs: list(xs))
        score_key(tok, m, kind, pre, f'{tag}/{view}/ben_wc_calib', ev(WC_CALIB))
        score_key(tok, m, kind, pre, f'{tag}/{view}/ben_wc_test',  ev(WC_TEST))
        score_key(tok, m, kind, pre, f'{tag}/{view}/ben_hard',     ev(HARD))
        score_key(tok, m, kind, pre, f'{tag}/{view}/orig',         ev(EVAL))
        pool = score_key(tok, m, kind, pre, f'{tag}/{view}/pool',  ev(flat))
        store[f'{tag}/{view}/pool'] = pool.reshape(NS, NC); save()
    m = tok = None; import gc; gc.collect(); torch.cuda.empty_cache(); print('  done', tag, flush=True)

# MAIN RUN = L0 view only for both experts -> full headline (H1, H3, H4, H6). Resumable across sessions via the
# shard checkpoints, so a timeout just continues next run. E_LG raw view is a separate optional follow-up below.
score_expert('meta-llama/Llama-Guard-3-8B',          'llamaguard', 'E_LG', 0, views=('l0',))
score_expert('ibm-granite/granite-guardian-4.1-8b',  'granite41',  'E_GR', 0, views=('l0',))
print('experts scored (L0):', sorted({k.split("/")[0] for k in store if "/l0/" in k}))

In [ ]:
# Cell 3 (OPTIONAL, run only after the L0 main run is done) - E_LG raw-text view, for the expert-vs-L0 separation
# (secondary robustness check, prereg R3). Same checkpointing; safe to run in its own session.
score_expert('meta-llama/Llama-Guard-3-8B', 'llamaguard', 'E_LG', 0, views=('raw',))
print('E_LG raw view done:', 'E_LG/raw/pool' in store)